HuggingFace transformers inference

In [1]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "Qwen/Qwen3-0.6B"

# use half-precision inference and GPU if available:
# CUDA (NVIDIA) or MPS (Apple Silicon GPU)
if torch.cuda.is_available():
    device, dtype = "cuda", torch.bfloat16
elif torch.backends.mps.is_available():
    device, dtype = "mps", torch.float16
else:
    device, dtype = "cpu", torch.float16

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name, dtype=dtype, device_map=device)

# prepare the model input
prompt = "How important is LLMOps on scale 0-10?"
messages = [{"role": "user", "content": prompt}]
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=True,  # use thinking / reasoning?
)
model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

# conduct text completion
generated_ids = model.generate(
    **model_inputs,
    max_new_tokens=32768,  # if not provided, use max context 32k
)
output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist()

# parse only output, not thinking content
try:
    # find token 151668 "</think>"
    index = len(output_ids) - output_ids[::-1].index(151668)
except ValueError:
    index = 0

content = tokenizer.decode(output_ids[index:], skip_special_tokens=True).strip("\n")
print("Answer:\n", content)

Answer:
 LLMOps (Large Language Model Operations) is a critical component in the deployment and management of large language models (LLMs) across various industries. Its importance varies depending on the context and scale of the model, but here's a structured analysis:

1. **Scale (0-10)**:  
   - **0-3**: Minimal impact, often just a technical detail or a tool used for specific tasks (e.g., data preprocessing).  
   - **4-6**: Moderate importance, with LLMOps being a key factor in integration, performance, and user adoption.  
   - **7-9**: High importance, as LLMOps ensures scalability, reliability, and adaptability of models in real-world applications.  
   - **10**: Maximal importance, with LLMOps being the foundation for seamless, enterprise-level deployment and continuous improvement of LLMs.

2. **Factors Influencing Importance**:  
   - **Model Size**: Larger models (e.g., GPT-4) require more robust operations, but smaller models (e.g., Llama) can be more flexible.  
   - **In

In [ ]:
import torch
from transformers import pipeline

device = (
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)

model = pipeline(
    task="text-generation",
    model="Qwen/Qwen3-0.6B",
    dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float16,
    device=device,
)

prompt = "How important is LLMOps on scale 0-10?"
messages = [{"role": "user", "content": prompt}]

messages = model(messages, max_new_tokens=32768)[0]["generated_text"]
content = messages[-1]["content"].strip()
print("Answer:\n", content)